In [1]:
# 运行这个单元格安装依赖（大约需要 1~2 分钟）
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # 最大上下文长度
dtype = None # 自动检测
load_in_4bit = True # 开启 4bit，显存立省 70%

# 加载 Gemma 4 E4B 对话模型
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",  # 或者用 "unsloth/gemma-4-E4B-unsloth-bnb-4bit"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, # 必须开启 4bit，才能在 T4 的 15GB 显存中丝滑运行
)

# 配置 LoRA 微调参数（只训练一小部分权重）
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA 秩，越大拟合越强
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [3]:
from datasets import load_dataset

# 1. 自动从 Hugging Face 下载最经典的开源中文羊驼数据集（取前 5000 条，免认证、公开即下）
print("正在下载开源数据集 shibing624/alpaca-zh ...")
dataset = load_dataset("shibing624/alpaca-zh", split="train[:5000]")

# 2. 转换为 Gemma 官方标准对话模板格式
def format_prompts(batch):
    texts = []
    for inst, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        user_msg = f"{inst}\n{inp}".strip() if inp else inst
        # 组装成 Gemma 对话标签
        text = f"<start_of_turn>user\n{user_msg}<end_of_turn>\n<start_of_turn>model\n{out}<end_of_turn>"
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched = True)

print("开源数据加载成功！第一条样本内容：\n")
print(dataset[0]["text"])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        # 方案 A（推荐）：学完整整 1 遍（大约 625 步，T4 大约耗时 15~20 分钟）
        num_train_epochs = 1, 
        
        # 方案 B：如果赶时间，可以指定步数，比如跑 250 步（学约 2000 条，耗时 7~8 分钟）
        # max_steps = 250, 
        
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        output_dir = "outputs",
    ),
)

# 开始训练
trainer.train()

In [ ]:
# 1. 切换到推理加速模式
FastLanguageModel.for_inference(model)

# 2. 准备测试问题列表
test_questions = [
    "请用通俗易懂的语言向中学生解释什么是‘分布式系统’，并举一个生活中的例子。",
    "我今天计划慢跑 5 公里但外面下暴雨，请为我设计一套 30 分钟的室内替代训练方案（包含热身、核心和拉伸）。",
    "用 Go 语言写一个并发安全的计数器（Counter 结构体），并简要说明原理。",
    "鲁迅和周树人到底是什么关系？他们俩谁先发表了《狂人日记》？"
]

print("🤖 开始检验 Gemma 4 学习成果：\n" + "="*50)

for idx, q in enumerate(test_questions, 1):
    prompt = f"<start_of_turn>user\n{q}<end_of_turn>\n<start_of_turn>model\n"
    
    # 明确传入 text=[prompt]
    inputs = tokenizer(text = [prompt], return_tensors = "pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 256, 
        use_cache = True,
        temperature = 0.7,
    )
    
    # 提取回答文本
    reply = tokenizer.batch_decode(outputs)[0]
    # 清理掉开头的 prompt 标签，只看回答
    answer = reply.split("<start_of_turn>model\n")[-1].replace("<end_of_turn>", "")
    
    print(f"\n【问题 {idx}】：{q}")
    print(f"【Gemma 4 回答】：\n{answer.strip()}")
    print("-" * 50)

In [ ]:
import gc
import torch

# 1. 彻底销毁训练器和数据缓存，给 GGUF 转换子进程腾出宝贵的 6GB+ 内存！
print("正在清理内存...")
if 'trainer' in locals():
    del trainer
if 'dataset' in locals():
    del dataset

gc.collect()
torch.cuda.empty_cache()

# 2. 开始合并权重，并打包压制成单个 GGUF 文件（大约耗时 1~2 分钟）
print("开始打包单个 GGUF 文件（包含基座+你的训练成果）...")
model.save_pretrained_gguf(
    "my_gemma_model", 
    tokenizer, 
    quantization_method = "q4_k_m"
)

print("恭喜！单文件导出成功！")